In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
from pathlib import Path

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

data_path = project_root / "data" / "raw" / "heart_disease.csv"

print("Dataset path:", data_path)
print("Dataset exists:", data_path.exists())

df = pd.read_csv(data_path)

Dataset path: c:\Users\User\OneDrive\Desktop\Heart-Disease-Classification\data\raw\heart_disease.csv
Dataset exists: True


In [6]:
rows, columns = df.shape

print(f"Number of rows: {rows}")
print(f"Number of columns: {columns}")

Number of rows: 4238
Number of columns: 16


In [7]:
df.head()

,Gender,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,Heart_ stroke
0,Male,39,postgraduate,0,0.0,0.0,no,0,0,195.0,106.0,70.0,26.97,80.0,77.0,No
1,Female,46,primaryschool,0,0.0,0.0,no,0,0,250.0,121.0,81.0,28.73,95.0,76.0,No
2,Male,48,uneducated,1,20.0,0.0,no,0,0,245.0,127.5,80.0,25.34,75.0,70.0,No
3,Female,61,graduate,1,30.0,0.0,no,1,0,225.0,150.0,95.0,28.58,65.0,103.0,yes
4,Female,46,graduate,1,23.0,0.0,no,0,0,285.0,130.0,84.0,23.10,85.0,85.0,No


In [8]:
print("Dataset columns:\n")

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

Dataset columns:

1. Gender
2. age
3. education
4. currentSmoker
5. cigsPerDay
6. BPMeds
7. prevalentStroke
8. prevalentHyp
9. diabetes
10. totChol
11. sysBP
12. diaBP
13. BMI
14. heartRate
15. glucose
16. Heart_ stroke


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4238 entries, 0 to 4237
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Gender           4238 non-null   str    
 1   age              4238 non-null   int64  
 2   education        4133 non-null   str    
 3   currentSmoker    4238 non-null   int64  
 4   cigsPerDay       4209 non-null   float64
 5   BPMeds           4185 non-null   float64
 6   prevalentStroke  4238 non-null   str    
 7   prevalentHyp     4238 non-null   int64  
 8   diabetes         4238 non-null   int64  
 9   totChol          4188 non-null   float64
 10  sysBP            4238 non-null   float64
 11  diaBP            4238 non-null   float64
 12  BMI              4219 non-null   float64
 13  heartRate        4237 non-null   float64
 14  glucose          3850 non-null   float64
 15  Heart_ stroke    4238 non-null   str    
dtypes: float64(8), int64(4), str(4)
memory usage: 529.9 KB


In [10]:
feature_overview = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Non-Null Count": df.notna().sum(),
    "Missing Count": df.isna().sum(),
    "Unique Values": df.nunique(dropna=False)
})

feature_overview

,Data Type,Non-Null Count,Missing Count,Unique Values
Gender,str,4238,0,2
age,int64,4238,0,39
education,str,4133,105,5
currentSmoker,int64,4238,0,2
cigsPerDay,float64,4209,29,34
BPMeds,float64,4185,53,3
prevalentStroke,str,4238,0,2
prevalentHyp,int64,4238,0,2
diabetes,int64,4238,0,2
totChol,float64,4188,50,249


In [11]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
})

missing_summary = (
    missing_summary[missing_summary["Missing Count"] > 0]
    .sort_values(by="Missing Count", ascending=False)
)

missing_summary

,Missing Count
glucose,388
education,105
BPMeds,53
totChol,50
cigsPerDay,29
BMI,19
heartRate,1


In [12]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


In [16]:
target_column = "Heart_ stroke"

target_clean = (
    df[target_column]
    .astype("string")
    .str.strip()
    .str.lower()
)

target_counts = target_clean.value_counts(dropna=False)

target_summary = (
    target_counts
    .rename_axis("Heart Disease Status")
    .reset_index(name="Count")
)

target_summary

,Heart Disease Status,Count
0,no,3594
1,yes,644


In [18]:
majority_class = target_counts.idxmax()
minority_class = target_counts.idxmin()

majority_count = target_counts.max()
minority_count = target_counts.min()

imbalance_ratio = majority_count / minority_count

print(f"Majority class: {majority_class}")
print(f"Majority class count: {majority_count}")
print(f"Minority class: {minority_class}")
print(f"Minority class count: {minority_count}")
print(f"Class imbalance ratio: {imbalance_ratio:.2f} : 1")

Majority class: no
Majority class count: 3594
Minority class: yes
Minority class count: 644
Class imbalance ratio: 5.58 : 1


In [19]:
numerical_summary = df.describe().T

numerical_summary

,count,mean,std,min,25%,50%,75%,max
age,4238.0,49.584946,8.572160,32.00,42.00,49.0,56.000,70.0
currentSmoker,4238.0,0.494101,0.500024,0.00,0.00,0.0,1.000,1.0
cigsPerDay,4209.0,9.003089,11.920094,0.00,0.00,0.0,20.000,70.0
BPMeds,4185.0,0.029630,0.169584,0.00,0.00,0.0,0.000,1.0
prevalentHyp,4238.0,0.310524,0.462763,0.00,0.00,0.0,1.000,1.0
diabetes,4238.0,0.025720,0.158316,0.00,0.00,0.0,0.000,1.0
totChol,4188.0,236.721585,44.590334,107.00,206.00,234.0,263.000,696.0
sysBP,4238.0,132.352407,22.038097,83.50,117.00,128.0,144.000,295.0
diaBP,4238.0,82.893464,11.910850,48.00,75.00,82.0,89.875,142.5
BMI,4219.0,25.802008,4.080111,15.54,23.07,25.4,28.040,56.8


In [20]:
categorical_columns = df.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

for column in categorical_columns:
    print(f"\nColumn: {column}")

    category_summary = (
        df[column]
        .value_counts(dropna=False)
        .rename_axis("Category")
        .reset_index(name="Count")
    )

    category_summary["Percentage"] = (
        category_summary["Count"] / len(df) * 100
    ).round(2)

    display(category_summary)

Categorical columns:
['Gender', 'education', 'prevalentStroke', 'Heart_ stroke']

Column: Gender


,Category,Count,Percentage
0,Female,2419,57.08
1,Male,1819,42.92



Column: education


,Category,Count,Percentage
0,uneducated,1720,40.59
1,primaryschool,1253,29.57
2,graduate,687,16.21
3,postgraduate,473,11.16
4,NaN,105,2.48



Column: prevalentStroke


,Category,Count,Percentage
0,no,4213,99.41
1,yes,25,0.59



Column: Heart_ stroke


,Category,Count,Percentage
0,No,3594,84.8
1,yes,644,15.2


In [21]:
low_cardinality_columns = [
    column
    for column in df.select_dtypes(include=np.number).columns
    if df[column].nunique(dropna=True) <= 10
]

print("Low-cardinality numerical columns:")
print(low_cardinality_columns)

for column in low_cardinality_columns:
    value_summary = (
        df[column]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("Value")
        .reset_index(name="Count")
    )

    value_summary["Percentage"] = (
        value_summary["Count"] / len(df) * 100
    ).round(2)

    print(f"\nColumn: {column}")
    display(value_summary)

Low-cardinality numerical columns:
['currentSmoker', 'BPMeds', 'prevalentHyp', 'diabetes']

Column: currentSmoker


,Value,Count,Percentage
0,0,2144,50.59
1,1,2094,49.41



Column: BPMeds


,Value,Count,Percentage
0,0.0,4061,95.82
1,1.0,124,2.93
2,NaN,53,1.25



Column: prevalentHyp


,Value,Count,Percentage
0,0,2922,68.95
1,1,1316,31.05



Column: diabetes


,Value,Count,Percentage
0,0,4129,97.43
1,1,109,2.57
